In [6]:
import pandas as pd

# Cargar el dataset
# Guardar en forma de tabla: 'sep' indica el separador de columnas, 'encoding' indica la codificación del archivo
df = pd.read_csv('data/unificado/modis_2024_unificado.csv', sep=',', encoding='utf-8') 

print(df.head())

   latitude  longitude  brightness  scan  track    acq_date  acq_time  \
0   40.8143    19.6998       300.5   1.1    1.1  2024-01-28       853   
1   40.4384    19.7503       319.6   1.0    1.0  2024-01-28      1218   
2   40.2651    19.8576       302.3   2.1    1.4  2024-01-31      1244   
3   39.8510    20.2340       301.5   1.0    1.0  2024-02-04      1213   
4   40.5204    20.3950       304.2   1.0    1.0  2024-02-04      1213   

  satellite instrument  confidence  version  bright_t31   frp daynight  type  \
0     Terra      MODIS          37    61.03       280.8   7.9        D     0   
1      Aqua      MODIS          79    61.03       285.4  20.4        D     2   
2      Aqua      MODIS          49    61.03       286.7  19.0        D     0   
3      Aqua      MODIS          35    61.03       288.9   5.3        D     0   
4      Aqua      MODIS          46    61.03       288.5   6.3        D     0   

      pais  
0  Albania  
1  Albania  
2  Albania  
3  Albania  
4  Albania  


In [7]:
# Eliminar filas duplicadas
df = df.drop_duplicates()

# Limpiar los datos eliminando filas con valores nulos
df = df.dropna()

In [10]:
# Convertir la columna 'fecha' a tipo datetime
df['acq_date'] = pd.to_datetime(df['acq_date'], format='%Y-%m-%d')

# Formatear 'acq_time' para que sea consistente con 0 a la izquierda
df['acq_time'] = df['acq_time'].astype(str).str.zfill(4)

# Creamos columna 'acq_datetime' combinando 'acq_date' y 'acq_time'
df['acq_datetime'] = pd.to_datetime(
    df['acq_date'].dt.strftime('%Y-%m-%d') + ' ' + df['acq_time'].str[:2] + ':' + df['acq_time'].str[2:4], format='%Y-%m-%d %H:%M')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308402 entries, 0 to 308401
Data columns (total 17 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   latitude      308402 non-null  float64       
 1   longitude     308402 non-null  float64       
 2   brightness    308402 non-null  float64       
 3   scan          308402 non-null  float64       
 4   track         308402 non-null  float64       
 5   acq_date      308402 non-null  datetime64[ns]
 6   acq_time      308402 non-null  object        
 7   satellite     308402 non-null  object        
 8   instrument    308402 non-null  object        
 9   confidence    308402 non-null  int64         
 10  version       308402 non-null  float64       
 11  bright_t31    308402 non-null  float64       
 12  frp           308402 non-null  float64       
 13  daynight      308402 non-null  object        
 14  type          308402 non-null  int64         
 15  pais          308

In [13]:
# Filtrar registros de coordenadas
# Latitud debe estar entre -90 y 90, longitud entre -180 y 180
df = df[
    (df['latitude'].between(-90, 90)) &
    (df['longitude'].between(-180, 180)) &
    (df['latitude'].notnull()) &
    (df['longitude'].notnull())
]

df = df[df['confidence'].between(0, 100)]

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308402 entries, 0 to 308401
Data columns (total 17 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   latitude      308402 non-null  float64       
 1   longitude     308402 non-null  float64       
 2   brightness    308402 non-null  float64       
 3   scan          308402 non-null  float64       
 4   track         308402 non-null  float64       
 5   acq_date      308402 non-null  datetime64[ns]
 6   acq_time      308402 non-null  object        
 7   satellite     308402 non-null  object        
 8   instrument    308402 non-null  object        
 9   confidence    308402 non-null  int64         
 10  version       308402 non-null  float64       
 11  bright_t31    308402 non-null  float64       
 12  frp           308402 non-null  float64       
 13  daynight      308402 non-null  object        
 14  type          308402 non-null  int64         
 15  pais          308

In [ ]:
# Convertir variables categóricas para optimizar memoria 
df['satellite'] = df['satellite'].astype('category')
df['instrument'] = df['instrument'].astype('category')
df['daynight'] = df['daynight'].astype('category')
df['pais'] = df['pais'].astype('category')
df['type'] = df['type'].astype('category') # tipo de foco (forestal, volcán...)

# Redondear variables numéricas para homogeneizar datos y reducir memoria
df['brightness'] = df['brightness'].round(2) # temperatura de brillo del pixel medida en Kelvin (temperatura observada desde el espacio en la zona más caliente)
df['bright_t31'] = df['bright_t31'].round(2) # temperatura de brillo del pixel medida en infra-rojo (band 31) (temperatura de la superficie terrestre)
# brightness y bright_t31 se usan para contrastar y distinguir un incendio de otros objetos calientes (asfalto, volcanes)
df['frp'] = df['frp'].round(2) # fire radiative power (potencia radiactiva del fuego) medida en MW (megavatios). Es un indicador de la intensidad del incendio.

Si quieres analizar la gravedad o potencia del incendio $\rightarrow$ Usa frp.

Un frp bajo (5-15 MW) suele ser un foco pequeño o quema controlada; un frp alto (>100 o miles de MW) indica un incendio de gran intensidad.

Si quieres analizar la temperatura máxima aparente $\rightarrow$ Usa brightness.

Si ves que brightness es muy alto pero bright_t31 también lo es de forma homogénea en la zona, suele ser indicador de temperaturas ambientales del suelo muy altas, mientras que una diferencia drástica entre ambas confirma la presencia de llamas ardientes.

In [20]:
# Diccionario de mapeo oficial de MODIS
diccionario_tipos = {
    0: 'Incendio forestal/vegetación',
    1: 'Volcán',
    2: 'Industria/Foco estático',
    3: 'Otros/Fuera de rango'
}

# Aplicar el mapeo a la columna type
df['type'] = df['type'].map(diccionario_tipos)

# Asegurarnos de que se guarde como tipo categoría para optimizar
df['type'] = df['type'].astype('category')

# Verificar el cambio
print(df['type'].value_counts())

Series([], Name: count, dtype: int64)


In [21]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308402 entries, 0 to 308401
Data columns (total 17 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   latitude      308402 non-null  float64       
 1   longitude     308402 non-null  float64       
 2   brightness    308402 non-null  float64       
 3   scan          308402 non-null  float64       
 4   track         308402 non-null  float64       
 5   acq_date      308402 non-null  datetime64[ns]
 6   acq_time      308402 non-null  object        
 7   satellite     308402 non-null  object        
 8   instrument    308402 non-null  object        
 9   confidence    308402 non-null  int64         
 10  version       308402 non-null  float64       
 11  bright_t31    308402 non-null  float64       
 12  frp           308402 non-null  float64       
 13  daynight      308402 non-null  object        
 14  type          0 non-null       category      
 15  pais          308